# Checkpoint and Resume with AgenticWorkflow

This notebook demonstrates the checkpoint/resume system in `AgenticWorkflow`. When processing large datasets, you can:

1. Save progress automatically after each item
2. Resume from where you left off if interrupted
3. Inspect the working directory for detailed logs and metadata

> **All cells in this notebook require a real LLM API key** (e.g., `OPENAI_API_KEY`). The checkpoint system saves real results to disk.

## Setup

In [ ]:
import pandas as pd
from pathlib import Path
from lattereview.agentic import ScoringReviewer, AgenticWorkflow

# Sample dataset
data = pd.DataFrame({
    "text": [
        "Deep learning for retinal disease detection achieved 96% accuracy on 50,000 fundus images.",
        "A survey of 150 clinicians found 62% support AI-assisted diagnosis in radiology.",
        "Federated learning enabled multi-site brain tumor segmentation without data sharing, matching centralized performance.",
        "NLP pipeline extracted medication information from 100,000 clinical notes with 89% F1 score.",
        "Reinforcement learning optimized radiation therapy planning, reducing treatment time by 30%.",
    ]
})

WORKING_DIR = Path("./review_output")
print(f"Working directory: {WORKING_DIR.resolve()}")
print(f"Dataset size: {len(data)} items")

## First Run: Process Items with Checkpointing

The workflow saves results atomically after each item completes. If the process is interrupted, completed items are preserved.

> **Requires API key:** Set `OPENAI_API_KEY` before running.

In [ ]:
scorer = ScoringReviewer(
    name="QualityScorer",
    backstory="You are a research quality evaluator for medical AI studies.",
    model="openai:gpt-5.4-mini",
    scoring_task="Rate the scientific rigor of this study.",
    scoring_set=[1, 2, 3, 4, 5],
    scoring_rules="1=anecdotal, 2=weak, 3=adequate, 4=strong, 5=excellent",
    max_iterations=1,
)

# First run — processes all items, saves checkpoints
workflow = AgenticWorkflow(
    workflow_schema=[
        {
            "round": "A",
            "reviewers": [scorer],
            "text_inputs": ["text"],
        }
    ],
    working_dir=WORKING_DIR,
    verbose=True,
)

result_df = await workflow(data)
print(f"\nProcessed {len(result_df)} items")
result_df.head()

## Resume: Skip Already-Completed Items

Set `resume=True` to load previous results and skip items that were already processed. Only new or unfinished items will be reviewed.

This is useful when:
- A long run was interrupted (network error, timeout, etc.)
- You want to add more items to an existing dataset
- You need to re-run with a modified reviewer on only the remaining items

In [ ]:
# Resume — will detect that all 5 items are already done and skip them
workflow_resumed = AgenticWorkflow(
    workflow_schema=[
        {
            "round": "A",
            "reviewers": [scorer],
            "text_inputs": ["text"],
        }
    ],
    working_dir=WORKING_DIR,
    resume=True,  # Load existing checkpoints
    verbose=True,
)

result_df_resumed = await workflow_resumed(data)
print(f"\nResumed: {len(result_df_resumed)} items (all previously completed)")

In [ ]:
# Adding new items and resuming — only the new items get processed
expanded_data = pd.concat([
    data,
    pd.DataFrame({"text": [
        "A transformer model predicted ICU mortality with AUC=0.91 on 200,000 patient records.",
    ]})
], ignore_index=True)

workflow_expanded = AgenticWorkflow(
    workflow_schema=[
        {
            "round": "A",
            "reviewers": [scorer],
            "text_inputs": ["text"],
        }
    ],
    working_dir=WORKING_DIR,
    resume=True,
    verbose=True,
)

result_df_expanded = await workflow_expanded(expanded_data)
print(f"\nTotal items: {len(result_df_expanded)} (only 1 new item processed)")

## Working Directory Structure

After a workflow run, the working directory contains:

```
review_output/
├── run_metadata.json              # RunState: progress, config hash, costs
├── round_A/
│   └── agent_QualityScorer/
│       ├── memory/                # Agent memory store
│       │   └── _index.json
│       ├── flags/                 # Flagged items
│       │   └── flags.json
│       ├── logs/                  # Per-item action logs
│       │   └── item_0.jsonl
│       └── results/               # Per-item structured output
│           └── item_0.json
└── output/
    ├── after_round_A.parquet      # DataFrame snapshot after round
    └── final.parquet              # Complete merged DataFrame
```

Each item's result is saved atomically (write-to-temp-then-rename) as soon as it completes, ensuring no work is lost on interruption.

In [ ]:
# Inspect the working directory
import os

if WORKING_DIR.exists():
    for root, dirs, files in os.walk(WORKING_DIR):
        level = root.replace(str(WORKING_DIR), "").count(os.sep)
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        sub_indent = "  " * (level + 1)
        for file in files:
            print(f"{sub_indent}{file}")
else:
    print("Working directory not yet created (run the workflow first).")